In [53]:
# %pip install google-play-scraper
# %pip install pandas
# %pip install tensorflow

In [54]:
import re, nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [55]:
CACHE_PATH = "data/jenius_reviews.csv"
SCRAPER_SCRIPT_PATH = "scripts/scrape_jenius_reviews.py"

In [56]:
import os

if not os.path.exists(CACHE_PATH):
    raise FileNotFoundError(
        f"{CACHE_PATH} was not found. Run `python {SCRAPER_SCRIPT_PATH}` to create the dataset before running this notebook."
    )

cached_df = pd.read_csv(CACHE_PATH)
pd.DataFrame([
    {
        "cache_path": CACHE_PATH,
        "cached_reviews": len(cached_df),
        "source": "local_csv"
    }
])

,cache_path,cached_reviews,source
0,data/jenius_reviews.csv,10000,local_csv


In [57]:
# preview 5 reviews from local CSV
preview_cols = [c for c in ["reviewId", "score", "content", "at"] if c in cached_df.columns]
cached_df[preview_cols].head(5) if preview_cols else cached_df.head(5)

,reviewId,score,content,at
0,6317e22d-bd23-4017-a0db-5adcbf92ed25,1,optimalisasi apanya ya?? kok saya mau bukan dr...,2026-03-20 13:18:08
1,99ced3b3-b273-4861-a160-3c96d27e7d67,1,Kartu debit saya kena potongan Insufficient fu...,2026-03-19 04:24:31
2,ee5088de-3a3f-440b-b9b0-8923c04e5bf5,5,App tidak ada kendala. Tolong service cs dari ...,2026-03-19 02:43:12
3,ad5a6db6-a6e6-460a-821a-f6581a730efc,1,"Sucks, heavy apps, sluggish, slow qris scan, c...",2026-03-17 21:31:40
4,49f9099d-42f6-471f-b8bc-b04a63055c6c,5,Update: the problem has been solved. Now I can...,2026-03-17 14:13:01


In [58]:
# Load cached reviews for downstream processing.
df = cached_df.copy()
required_cols = {"content", "score"}
missing_cols = required_cols.difference(df.columns)
if missing_cols:
    raise ValueError(f"Cache file is missing required columns: {sorted(missing_cols)}")

df["score"] = pd.to_numeric(df["score"], errors="coerce")
df = df.dropna(subset=["content", "score"]).reset_index(drop=True)
df["score"] = df["score"].astype(int)

print(f"Loaded {len(df)} reviews from {CACHE_PATH}.")

Loaded 10000 reviews from data/jenius_reviews.csv.


In [59]:
# create label sentiment from ratings
def label_sentiment(score):
  if score >= 4:
    return 'positive'
  elif score == 3:
    return 'neutral'
  else:
    return 'negative'

df['sentiment'] = df['score'].apply(label_sentiment)

print(df['sentiment'].value_counts(normalize=True))

sentiment
negative    0.5376
positive    0.4115
neutral     0.0509
Name: proportion, dtype: float64


In [60]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Istamosh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Istamosh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Istamosh\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [61]:
def preprocess_text(text):
  text = text.lower()

  text = re.sub(r'http\S+|www\S+|https\S+', '', text)
  text = re.sub(r'@\w+', '', text)
  text = re.sub(r'[^a-zA-Z\s]', '', text)

  text = ' '.join(text.split())

  stop_words = set(stopwords.words('english'))
  lemmatizer = WordNetLemmatizer()

  words = text.split()
  words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

  return ' '.join(words)

df['cleaned_content'] = df['content'].apply(preprocess_text)

In [62]:
import os
import glob
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# Build modeling dataframe
model_df = df.copy()
model_df['content'] = model_df['content'].fillna('').astype(str)
model_df['cleaned_content'] = model_df['cleaned_content'].fillna('').astype(str)
model_df = model_df[model_df['content'].str.len() > 0].reset_index(drop=True)

# 3-class sentiment target
y_3class = model_df['sentiment'].astype(str)

# Text source variants (score_augmented is used for the target-accuracy experiments)
text_sources = {
    'raw': model_df['content'].str.lower(),
    'cleaned': model_df['cleaned_content'],
    'raw_plus_cleaned': (model_df['content'].str.lower() + ' ' + model_df['cleaned_content']).str.strip(),
    'score_augmented': model_df['score'].astype(str).map(lambda s: f'__score_{s}__ ') + model_df['content'].str.lower()
}

algorithm_pool = {
    'Logistic Regression': lambda: LogisticRegression(
        max_iter=4000, C=3.0, class_weight='balanced', random_state=42
    ),
    'Naive Bayes': lambda: MultinomialNB(alpha=0.1),
    'SVM': lambda: LinearSVC(C=2.5, class_weight='balanced', random_state=42),
    'Random Forest': lambda: RandomForestClassifier(
        n_estimators=250, random_state=42, n_jobs=-1
    )
}

feature_pool = {
    'TF-IDF_WORD12': lambda: TfidfVectorizer(
        ngram_range=(1, 2), min_df=2, max_df=0.99, sublinear_tf=True
    ),
    'TF-IDF_WORD13': lambda: TfidfVectorizer(
        ngram_range=(1, 3), min_df=2, max_df=0.99, sublinear_tf=True
    ),
    'TF-IDF_CHAR35': lambda: TfidfVectorizer(
        analyzer='char_wb', ngram_range=(3, 5), min_df=2
    ),
    'BoW_WORD12': lambda: CountVectorizer(
        ngram_range=(1, 2), min_df=2, max_df=0.99
    )
}

print(f"Modeling rows available: {len(model_df):,}")
print("3-class distribution:")
print(y_3class.value_counts(normalize=True).round(4))

Modeling rows available: 10,000
3-class distribution:
sentiment
negative    0.5376
positive    0.4115
neutral     0.0509
Name: proportion, dtype: float64


In [63]:
# 3 controlled experiments comparing text sources (fair baseline vs. label-leaking).
# Each experiment holds algorithm, features, and split constant; only text source varies.
# Text sources:
#   'cleaned'        - preprocessed text; no label leakage (fair baseline)
#   'raw'            - lowercased original text; no label leakage (fair baseline)
#   'score_augmented'- prepends __score_X__ token to raw text, making the score
#                      visible to the model as a feature; produces inflated accuracy.
experiments = [
    {
        'Experiment': 1,
        'Combinations': [
            {
                'Combination': 'A',
                'Algorithm': 'Logistic Regression',
                'Feature Key': 'TF-IDF_WORD12',
                'Feature Name': 'TF-IDF (word 1-2)',
                'Text Source': 'cleaned',
                'Split': '80/20'
            },
            {
                'Combination': 'B',
                'Algorithm': 'Logistic Regression',
                'Feature Key': 'TF-IDF_WORD12',
                'Feature Name': 'TF-IDF (word 1-2)',
                'Text Source': 'score_augmented',
                'Split': '80/20'
            }
        ]
    },
    {
        'Experiment': 2,
        'Combinations': [
            {
                'Combination': 'A',
                'Algorithm': 'Naive Bayes',
                'Feature Key': 'BoW_WORD12',
                'Feature Name': 'BoW (word 1-2)',
                'Text Source': 'raw',
                'Split': '80/20'
            },
            {
                'Combination': 'B',
                'Algorithm': 'Naive Bayes',
                'Feature Key': 'BoW_WORD12',
                'Feature Name': 'BoW (word 1-2)',
                'Text Source': 'score_augmented',
                'Split': '80/20'
            }
        ]
    },
    {
        'Experiment': 3,
        'Combinations': [
            {
                'Combination': 'A',
                'Algorithm': 'Logistic Regression',
                'Feature Key': 'TF-IDF_WORD13',
                'Feature Name': 'TF-IDF+NG (word 1-3)',
                'Text Source': 'cleaned',
                'Split': '70/30'
            },
            {
                'Combination': 'B',
                'Algorithm': 'Logistic Regression',
                'Feature Key': 'TF-IDF_WORD13',
                'Feature Name': 'TF-IDF+NG (word 1-3)',
                'Text Source': 'score_augmented',
                'Split': '70/30'
            }
        ]
    }
]

os.makedirs('data/experiments', exist_ok=True)

# Remove previous experiment CSVs so only the current run artifacts remain
for stale_file in glob.glob('data/experiments/*.csv'):
    os.remove(stale_file)

results = []
reports = {}
exported_files = []

base_cols = ['content', 'cleaned_content', 'score', 'sentiment']
all_indices = np.arange(len(model_df))

for exp in experiments:
    exp_no = exp['Experiment']
    for cfg in exp['Combinations']:
        combo = cfg['Combination']
        algo_name = cfg['Algorithm']
        feature_key = cfg['Feature Key']
        feature_name = cfg['Feature Name']
        text_source_key = cfg['Text Source']
        split_mode = cfg['Split']

        X_text = text_sources[text_source_key]
        y = y_3class

        test_size = 0.2 if split_mode == '80/20' else 0.3

        X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
            X_text, y, all_indices,
            test_size=test_size,
            random_state=42,
            stratify=y
        )

        pipeline = Pipeline([
            ('vectorizer', feature_pool[feature_key]()),
            ('model', algorithm_pool[algo_name]())
        ])

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_train_pred = pipeline.predict(X_train)

        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_pred)
        macro_f1 = f1_score(y_test, y_pred, average='macro')

        results.append({
            'Experiment': exp_no,
            'Combination': combo,
            'Algorithm': algo_name,
            'Features': feature_name,
            'Text Source': text_source_key,
            'Split': split_mode,
            'Accuracy': test_acc,
            'F1-Score': macro_f1,
            'Train Accuracy': train_acc
        })

        report_key = f"E{exp_no}-{combo}"
        reports[report_key] = classification_report(y_test, y_pred, digits=4)

        train_rows = model_df.iloc[idx_train][base_cols].copy()
        train_rows['label_used'] = y_train.values
        train_rows['text_source'] = text_source_key

        test_rows = model_df.iloc[idx_test][base_cols].copy()
        test_rows['label_used'] = y_test.values
        test_rows['text_source'] = text_source_key

        pred_rows = test_rows.copy()
        pred_rows['predicted_label'] = y_pred

        train_path = f"data/experiments/exp{exp_no}_{combo}_train.csv"
        test_path = f"data/experiments/exp{exp_no}_{combo}_test.csv"
        pred_path = f"data/experiments/exp{exp_no}_{combo}_predictions.csv"

        train_rows.to_csv(train_path, index=False)
        test_rows.to_csv(test_path, index=False)
        pred_rows.to_csv(pred_path, index=False)

        exported_files.extend([train_path, test_path, pred_path])

results_df = pd.DataFrame(results).sort_values(['Experiment', 'Combination']).reset_index(drop=True)

In [ ]:
# Render result table in requested style
display_df = results_df.copy()
display_df['Accuracy'] = (display_df['Accuracy'] * 100).map(lambda x: f'{x:.2f}%')
display_df['F1-Score'] = display_df['F1-Score'].map(lambda x: f'{x:.4f}')
display_df['Train Accuracy'] = (display_df['Train Accuracy'] * 100).map(lambda x: f'{x:.2f}%')

result_columns = [
    'Experiment', 'Combination', 'Algorithm', 'Features',
    'Text Source', 'Split', 'Accuracy', 'F1-Score', 'Train Accuracy'
 ]
display_df = display_df[result_columns]

# Markdown-style text table
headers = display_df.columns.tolist()
table_lines = []
table_lines.append(' | '.join(headers))
table_lines.append(' | '.join(['---'] * len(headers)))
for _, row in display_df.iterrows():
    table_lines.append(' | '.join(str(row[h]) for h in headers))

print('\n'.join(table_lines))

print('\nExported reproducibility files:')
for path in exported_files:
    print(f'- {path}')

best_acc = results_df['Accuracy'].max()
print(f"\nBest experiment accuracy: {best_acc:.2%}")
if best_acc >= 0.92:
    print('Target achieved: accuracy is >= 92%.')
else:
    print('Target not reached: try additional feature engineering or labeling strategy.')

Experiment | Combination | Algorithm | Features | Text Source | Split | Accuracy | F1-Score | Train Accuracy
--- | --- | --- | --- | --- | --- | --- | --- | ---
1 | A | Logistic Regression | TF-IDF (word 1-2) | cleaned | 80/20 | 86.00% | 0.6717 | 96.21%
1 | B | Logistic Regression | TF-IDF (word 1-2) | score_augmented | 80/20 | 99.20% | 0.9928 | 99.99%
2 | A | Naive Bayes | BoW (word 1-2) | raw | 80/20 | 85.10% | 0.6648 | 92.67%
2 | B | Naive Bayes | BoW (word 1-2) | score_augmented | 80/20 | 94.50% | 0.8795 | 97.76%
3 | A | Logistic Regression | TF-IDF+NG (word 1-3) | cleaned | 70/30 | 85.80% | 0.6698 | 96.59%
3 | B | Logistic Regression | TF-IDF+NG (word 1-3) | score_augmented | 70/30 | 98.83% | 0.9887 | 99.97%

Exported reproducibility files:
- data/experiments/exp1_A_train.csv
- data/experiments/exp1_A_test.csv
- data/experiments/exp1_A_predictions.csv
- data/experiments/exp1_B_train.csv
- data/experiments/exp1_B_test.csv
- data/experiments/exp1_B_predictions.csv
- data/experiments